In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=== RANDOM FOREST v2 - COLDTRACK (Versión Mejorada) ===\n")

# Cargar datos
X = pd.read_csv("X_features.csv")
y = pd.read_csv("y_target.csv").squeeze()

print(f"Dataset: {X.shape[0]:,} registros × {X.shape[1]} features")

# ====================== ELIMINAMOS FEATURES QUE CAUSAN LEAKAGE ======================
features_to_remove = [
    'high_temp', 
    'critical_vibration', 
    'compressor_overwork', 
    'long_door_open'
]

X_clean = X.drop(columns=features_to_remove, errors='ignore')

print(f"Features después de limpieza: {X_clean.shape[1]} columnas")

# ====================== ENTRENAMIENTO CON VALIDACIÓN TEMPORAL ======================
tscv = TimeSeriesSplit(n_splits=5)

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,                    # Reducimos profundidad para evitar overfitting
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Último split para evaluación
train_index, test_index = list(tscv.split(X_clean))[-1]
X_train, X_test = X_clean.iloc[train_index], X_clean.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]

print(f"Train: {len(X_train):,} registros | Test: {len(X_test):,} registros")

# Entrenamiento
model.fit(X_train, y_train)

# Predicción
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# ====================== RESULTADOS ======================
print("\n" + "="*60)
print("RESULTADOS DEL MODELO v2")
print("="*60)
print(f"Accuracy     : {y_pred.mean():.4f}")
print(f"ROC AUC      : {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Mantenimiento (1)']))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

# ====================== IMPORTANCIA DE FEATURES ======================
importancia = pd.DataFrame({
    'Feature': X_clean.columns,
    'Importancia': model.feature_importances_
}).sort_values('Importancia', ascending=False)

print("\nTop 10 Features más importantes:")
print(importancia.head(10))

# ====================== GUARDAR MODELO ======================
joblib.dump(model, "coldtrack_rf_model_v2.pkl")
joblib.dump(X_clean.columns.tolist(), "feature_columns_v2.pkl")

print("\n✅ Modelo mejorado guardado como: coldtrack_rf_model_v2.pkl")
print("✅ Columnas guardadas como: feature_columns_v2.pkl")

ModuleNotFoundError: No module named 'pandas'